# Unsloth GRPO Fine-Tuning Demo
**Model:** `unsloth/Phi-3-mini-4k-instruct-bnb-4bit`  
**Method:** GRPO (Group Relative Policy Optimization) — a lightweight RL fine-tuning algorithm

## What this notebook does
1. Loads a 4-bit quantized Phi-3-mini model with LoRA adapters
2. Downloads 64 examples from GSM8K (grade-school math) as training data
3. Trains using GRPO: the model generates multiple answers per question, reward functions score each answer, and the model is nudged toward high-reward behaviors
4. Evaluates reasoning quality on 16 held-out test problems

## Why GRPO instead of SFT?
- SFT requires labeled "correct" responses. GRPO only needs a **verifiable reward signal** (e.g. "is the answer correct?")
- GRPO is the algorithm behind DeepSeek-R1 and similar "reasoning model" training pipelines
- GRPO is cheaper than PPO — no separate value/critic model needed

## How GRPO works (one training step)
1. Take one prompt (a math question)
2. Generate **N completions** from the current policy
3. Score each completion with reward functions (correctness, format, length...)
4. Compute **relative advantages**: completions above the group average are reinforced, below are suppressed
5. Update the model via a clipped policy gradient (like PPO, but without a critic)

## Required output format
The model must reply with:
```xml
<reasoning>
step-by-step reasoning...
</reasoning>
<answer>
final numeric answer
</answer>
```

## Step 1 — Check GPU

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())
print("GPU count:      ", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:       ", torch.cuda.get_device_name(0))
    print("VRAM (GB):      ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

PyTorch version: 2.10.0+cu130
CUDA available:  True
GPU count:       1
GPU name:        NVIDIA GeForce RTX 5070 Ti Laptop GPU
VRAM (GB):       12.8


## Step 2 — Import Libraries
`PatchFastRL` rewrites TRL's GRPOTrainer internals to use Unsloth's optimized kernels.

In [2]:
import re
from unsloth import FastLanguageModel, PatchFastRL
from trl import GRPOConfig, GRPOTrainer
from datasets import load_dataset

# Patch TRL's GRPOTrainer to use Unsloth's fast kernels.
# Must be called before loading the model.
PatchFastRL("GRPO", FastLanguageModel)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


D:\Software\Anaconda\envs\unsloth_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0514 16:44:25.204000 16152 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.


## Step 3 — Load the Base Model (4-bit)

In [3]:
# 1024 tokens is enough for GSM8K questions + reasoning chains in this demo.
# Increase to 2048 if you encounter truncation.
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    # FP16 is safer than bfloat16 on older GPUs; bfloat16 is fine on Ampere+.
    dtype=torch.float16,
    # 4-bit NF4 quantization: cuts VRAM from ~14 GB to ~6 GB.
    load_in_4bit=True,
    # fast_inference=False: keep the training-compatible forward path.
    fast_inference=False,
)

# Some models omit pad_token; use eos_token as a fallback.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded.")

==((====))==  Unsloth 2026.5.2: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5070 Ti Laptop GPU. Num GPUs = 1. Max memory: 11.94 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:03<00:00, 88.63it/s] 


Model loaded.


## Step 4 — Attach LoRA Adapters
Rank 32 is a good balance: enough capacity to learn structured reasoning without using too much VRAM.

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    # Rank: lower than QLoRA (64) because RL fine-tuning is noisier;
    # a smaller adapter is less likely to overfit.
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    # Convention: lora_alpha = 2 × r.
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)")

Unsloth 2026.5.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Trainable params: 59.8M / 2.07B (2.89%)


## Step 5 — Load and Format the GSM8K Dataset
GSM8K (Grade School Math 8K) is perfect for GRPO because:
- Each problem has an unambiguous numeric final answer → easy to verify with a reward function
- Problems require multi-step reasoning → encourages the model to use the `<reasoning>` section

We only use **64 training** + **16 test** examples to keep this demo fast.

In [5]:
# System prompt tells the model what format to use at every turn.
SYSTEM_PROMPT = """You are a helpful reasoning model.

Always answer in this exact format:

<reasoning>
Write your step-by-step reasoning here.
</reasoning>
<answer>
Write only the final numeric answer here (e.g. 42).
</answer>"""


def extract_gsm8k_answer(raw_answer: str) -> str:
    """
    Pull the final number out of a GSM8K answer string.

    GSM8K answers look like: "reasoning steps... #### 42"
    We only want the part after ####.
    """
    if "####" in raw_answer:
        return raw_answer.split("####")[-1].strip()
    return raw_answer.strip()


def format_for_grpo(example: dict) -> dict:
    """
    Convert a GSM8K example into the format GRPOTrainer expects:
      - 'prompt': list of chat messages (system + user)
      - 'answer': the ground-truth final answer string

    GRPOTrainer will tokenize 'prompt' and generate completions from it.
    The reward functions receive 'answer' as an extra keyword argument.
    """
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": example["question"]},
        ],
        "answer": extract_gsm8k_answer(example["answer"]),
    }


# Download a tiny slice — fast to download and fast to train on.
raw_train = load_dataset("openai/gsm8k", "main", split="train[:64]")
raw_test  = load_dataset("openai/gsm8k", "main", split="test[:16]")

train_dataset = raw_train.map(format_for_grpo, remove_columns=raw_train.column_names)
test_dataset  = raw_test.map(format_for_grpo,  remove_columns=raw_test.column_names)

print(f"Train: {len(train_dataset)} examples")
print(f"Test:  {len(test_dataset)} examples")
print("\nSample prompt:")
for msg in train_dataset[0]["prompt"]:
    print(f"  [{msg['role']}]: {msg['content'][:80]}...")
print(f"  [answer]: {train_dataset[0]['answer']}")

Train: 64 examples
Test:  16 examples

Sample prompt:
  [system]: You are a helpful reasoning model.

Always answer in this exact format:

<reason...
  [user]: Natalia sold clips to 48 of her friends in April, and then she sold half as many...
  [answer]: 72


## Step 6 — Helper Functions for Reward Calculation
These utilities are shared across all reward functions.

In [6]:
def get_completion_text(completion) -> str:
    """
    Extract the raw text from a TRL completion object.

    GRPOTrainer passes completions in various forms depending on the version:
      - A list of chat dicts: [{"role": "assistant", "content": "..."}]
      - A plain string
    This helper normalizes both cases.
    """
    if isinstance(completion, list):
        return completion[0].get("content", "")
    if isinstance(completion, dict):
        return completion.get("content", "")
    return str(completion)


def extract_answer(text: str) -> str:
    """
    Extract the model's final numeric answer from its output.

    Priority order:
    1. Content inside <answer>...</answer> tags
    2. Content after #### (GSM8K-style fallback)
    3. Last number found in the text
    4. The whole (lowercased) text as a last resort
    """
    if not text:
        return ""

    text = str(text).strip()

    # 1. Try <answer>...</answer>
    tag_match = re.search(r"<answer>\s*(.*?)\s*</answer>", text, re.DOTALL | re.IGNORECASE)
    if tag_match:
        text = tag_match.group(1).strip()

    # 2. Fallback: take text after ####
    elif "####" in text:
        text = text.split("####")[-1].strip()

    # Remove currency symbols and commas so "$1,234" becomes "1234".
    cleaned = re.sub(r"[$€£¥,%]", "", text).strip()

    # 3. Return the last number found (handles "The answer is 42 dollars").
    numbers = re.findall(r"-?\d+(?:\.\d+)?(?:/\d+(?:\.\d+)?)?", cleaned)
    if numbers:
        return numbers[-1]

    # 4. Last resort: normalized text.
    return cleaned.lower()


def normalize_answer(text: str) -> str:
    """Lowercase, strip whitespace, and remove trailing punctuation for comparison."""
    return str(text).lower().strip().replace(" ", "").rstrip(".")


# Quick sanity check
assert extract_answer("<answer>\n$18\n</answer>") == "18"
assert normalize_answer(" 42 ") == "42"
print("Helper functions OK.")

Helper functions OK.


## Step 7 — Define Reward Functions
GRPO uses **multiple reward signals** combined additively. Each function returns one scalar per completion.

| Reward function | Max reward | What it measures |
|---|---|---|
| `correctness_reward` | +2.0 | Final answer matches ground truth |
| `format_reward` | +1.0 | XML tags are present and in correct order |
| `concise_answer_reward` | +0.5 | Answer is short (1–5 words) |
| `reasoning_length_reward` | +0.5 | Reasoning section has ≥5 words |

**Tip:** Correctness carries the most weight (+2.0) so the model prioritizes getting the right answer. The other rewards shape *how* the answer is expressed.

In [7]:
def correctness_reward(prompts, completions, answer, **kwargs):
    """
    +2.0 if the model's extracted answer matches the ground truth.
    This is the primary learning signal.

    GRPOTrainer passes dataset columns as keyword arguments,
    so 'answer' maps directly to the 'answer' column we set up.
    """
    rewards = []
    for completion, gold in zip(completions, answer):
        pred = normalize_answer(extract_answer(get_completion_text(completion)))
        gold = normalize_answer(gold)
        rewards.append(2.0 if pred == gold else 0.0)
    return rewards


def format_reward(completions, **kwargs):
    """
    +1.0 if the output strictly follows the required XML format:
    <reasoning>...</reasoning><answer>...</answer>

    This reward prevents the model from gaming correctness_reward
    by outputting a bare number without any reasoning.
    """
    # Anchored regex: both tags must be present and in order.
    pattern = r"^\s*<reasoning>\s*.*?\s*</reasoning>\s*<answer>\s*.*?\s*</answer>\s*$"
    rewards = []
    for completion in completions:
        text = get_completion_text(completion)
        match = bool(re.match(pattern, text, re.DOTALL))
        rewards.append(1.0 if match else 0.0)
    return rewards


def concise_answer_reward(completions, **kwargs):
    """
    +0.5 if the final answer is 1–5 words.

    Encourages compact final answers (e.g. "18" not "The answer is 18 dollars").
    A long <answer> section usually means the model is leaking reasoning into it.
    """
    rewards = []
    for completion in completions:
        ans = extract_answer(get_completion_text(completion))
        word_count = len(ans.split())
        rewards.append(0.5 if 1 <= word_count <= 5 else 0.0)
    return rewards


def reasoning_length_reward(completions, **kwargs):
    """
    +0.5 if the <reasoning> section contains at least 5 words.

    Prevents the model from producing empty or trivially short reasoning
    just to satisfy the format check.
    """
    rewards = []
    for completion in completions:
        text = get_completion_text(completion)
        match = re.search(r"<reasoning>\s*(.*?)\s*</reasoning>", text, re.DOTALL)
        if match and len(match.group(1).split()) >= 5:
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards


# --- Sanity check: a perfect response should score 4.0 total ---
perfect_completion = [[{
    "role": "assistant",
    "content": (
        "<reasoning>\n"
        "Janet's ducks lay 16 eggs/day. She eats 3 and bakes 4, so 16-7=9 eggs remain.\n"
        "9 eggs × $2 = $18.\n"
        "</reasoning>\n"
        "<answer>\n18\n</answer>"
    )
}]]

r_correct  = correctness_reward(None, perfect_completion, ["18"])[0]
r_format   = format_reward(perfect_completion)[0]
r_concise  = concise_answer_reward(perfect_completion)[0]
r_reasoning = reasoning_length_reward(perfect_completion)[0]

print(f"Correctness:     {r_correct}  (max 2.0)")
print(f"Format:          {r_format}  (max 1.0)")
print(f"Concise answer:  {r_concise}  (max 0.5)")
print(f"Reasoning length:{r_reasoning}  (max 0.5)")
print(f"Total:           {r_correct + r_format + r_concise + r_reasoning}  (max 4.0)")

Correctness:     2.0  (max 2.0)
Format:          1.0  (max 1.0)
Concise answer:  0.5  (max 0.5)
Reasoning length:0.5  (max 0.5)
Total:           4.0  (max 4.0)


## Step 8 — Configure GRPO Training
Key GRPO-specific settings:
- **`num_generations`**: How many completions to sample per prompt per step. More = better gradient estimate, but more VRAM and time. 3 is the minimum viable value.
- **`max_completion_length`**: Cap on how many tokens the model can generate. Keep short (128) for math to save VRAM.
- **`max_steps`**: 20 steps is tiny — enough to see the loss improve but not enough to fully converge. Increase to 200+ for real training.

In [8]:
training_args = GRPOConfig(
    output_dir="phi3-mini-grpo-gsm8k-tiny-demo",

    # Learning rate — lower than SFT because RL updates are noisier.
    learning_rate=5e-6,

    # Batch: 1 prompt per step, gradient accumulated over 9 steps.
    # Effective batch = 1 × 9 = 9 prompts per update.
    per_device_train_batch_size=1,
    gradient_accumulation_steps=9,

    # Generate 3 completions per prompt to estimate the group advantage.
    # Increasing this (e.g. to 8) improves quality but uses more VRAM.
    num_generations=3,

    # Limit prompt + completion lengths to fit in VRAM.
    max_prompt_length=512,
    max_completion_length=128,

    # 20 steps is just enough to observe learning; increase for real results.
    max_steps=30,

    logging_steps=1,
    save_steps=10,
    report_to="none",

    # Use FP16 (not BF16) — more compatible with older CUDA versions.
    fp16=True,
    bf16=False,

    gradient_checkpointing=True,
)

print("Training config ready.")

Training config ready.


## Step 9 — Create and Run the GRPOTrainer

In [9]:
trainer = GRPOTrainer(
    model=model,
    # In newer TRL versions the tokenizer arg is called 'processing_class'.
    processing_class=tokenizer,
    # All four reward functions are applied to every completion.
    # The total reward is their sum.
    reward_funcs=[
        correctness_reward,
        format_reward,
        concise_answer_reward,
        reasoning_length_reward,
    ],
    args=training_args,
    train_dataset=train_dataset,
)

print("Trainer created. Starting GRPO training...")
trainer.train()

Trainer created. Starting GRPO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 64 | Num Epochs = 2 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 9
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 9 x 1) = 9
 "-____-"     Trainable parameters = 59,768,832 of 3,880,848,384 (1.54% trained)
Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'disable_compile', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
D:\Software\Anaconda\envs\unsloth_env\Lib\site-packages\transformers\modeling_attn_mask_utils.

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / format_reward / mean,rewards / format_reward / std,rewards / concise_answer_reward / mean,rewards / concise_answer_reward / std,rewards / reasoning_length_reward / mean,rewards / reasoning_length_reward / std
1,0.000000,0.666667,0.288675,125.333336,104.000000,128.000000,0.888889,104.000000,104.000000,104.000000,0.000016,0.000000,0.000000,0.111111,0.333333,0.500000,0.000000,0.055556,0.166667
2,0.000000,1.111111,0.673575,123.222221,85.000000,128.000000,0.888889,85.000000,85.000000,85.000000,0.000010,0.444444,0.881917,0.111111,0.333333,0.500000,0.000000,0.055556,0.166667
3,0.000000,0.777778,0.481125,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,0.000012,0.222222,0.666667,0.000000,0.000000,0.500000,0.000000,0.055556,0.166667
4,0.000000,0.722222,0.384900,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,0.000008,0.222222,0.666667,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000
5,0.000000,0.500000,0.000000,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,0.000014,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000
6,0.000000,0.888889,0.346944,125.777779,108.000000,128.000000,0.888889,108.000000,108.000000,108.000000,0.000041,0.222222,0.666667,0.111111,0.333333,0.500000,0.000000,0.055556,0.166667
7,0.000000,0.944444,0.384900,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,0.000032,0.444444,0.881917,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000
8,0.000000,0.666667,0.288675,124.777779,99.000000,128.000000,0.888889,99.000000,99.000000,99.000000,0.000057,0.000000,0.000000,0.111111,0.333333,0.500000,0.000000,0.055556,0.166667
9,0.000000,0.555556,0.096225,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,0.000065,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.055556,0.166667
10,0.000000,0.500000,0.000000,128.000000,128.000000,128.000000,1.000000,0.000000,0.000000,0.000000,0.000097,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000


Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
D:\Software\Anaconda\envs\unsloth_env\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
D:\Software\Anaconda\envs\unsloth_env\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWar

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

TrainOutput(global_step=30, training_loss=1.4260627319203262e-07, metrics={'train_runtime': 396.2848, 'train_samples_per_second': 0.681, 'train_steps_per_second': 0.076, 'total_flos': 0.0, 'train_loss': 1.4260627319203262e-07})

## Step 10 — Save the LoRA Adapter (Optional)

In [10]:
# Uncomment to save the adapter (only ~100 MB, not the full 3.8 B model).
# model.save_pretrained("phi3-mini-grpo-reasoning-lora")
# tokenizer.save_pretrained("phi3-mini-grpo-reasoning-lora")
# print("Saved to: phi3-mini-grpo-reasoning-lora")

print("(Save skipped — uncomment above lines to persist the adapter.)")

(Save skipped — uncomment above lines to persist the adapter.)


## Step 11 — Inference
Switch the model to fast inference mode and test it on questions it has never seen.

In [11]:
# Switch to inference mode: Unsloth enables faster forward-pass kernels.
FastLanguageModel.for_inference(model)

# Clear the pretrained generation_config's max_length so we can set max_new_tokens
# without the "both max_new_tokens and max_length are set" warning.
model.generation_config.max_length = None


def generate_answer(question: str, max_new_tokens: int = 256) -> str:
    """
    Run inference on a single math question and return the model's full output.

    Args:
        question: The math problem as a plain string.
        max_new_tokens: Maximum tokens to generate (256 is enough for most GSM8K answers).

    Returns:
        The model's raw text output (includes <reasoning> and <answer> tags).
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]

    # Tokenize the chat prompt.
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    # Build an explicit attention mask to avoid the "attention_mask not set" warning.
    attention_mask = torch.ones_like(input_ids)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        # Slight randomness helps produce varied reasoning chains.
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Strip the prompt tokens — keep only the newly generated tokens.
    new_tokens = outputs[:, input_ids.shape[1]:]
    return tokenizer.decode(new_tokens[0], skip_special_tokens=True)


print("Inference helper ready.")

Inference helper ready.


## Step 12 — Evaluate on Test Set

In [12]:
def get_question(sample: dict) -> str:
    """Pull the user-turn content out of a formatted prompt."""
    for msg in sample["prompt"]:
        if msg["role"] == "user":
            return msg["content"]
    return ""


# --- Single example walkthrough ---
sample = test_dataset[0]
question    = get_question(sample)
gold_answer = sample["answer"]
model_output = generate_answer(question)
pred_answer  = extract_answer(model_output)

print("Question:")
print(question)
print("\nModel output:")
print(model_output)
print("\nExtracted answer:", pred_answer)
print("Gold answer:     ", gold_answer)
print("Correct:", normalize_answer(pred_answer) == normalize_answer(gold_answer))

D:\Software\Anaconda\envs\unsloth_env\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
D:\Software\Anaconda\envs\unsloth_env\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
D:\Software\Anaconda\envs\unsloth_env\Lib\site-packages\transformers\modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be remove

Question:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Model output:
<reasoning>
To calculate the daily earnings from selling duck eggs at the farmers' market, we need to follow these steps:

1. Determine the total number of eggs laid by all ducks in a day.
2. Subtract the number of eggs Janet consumes herself.
3. Subtract the number of eggs used for baking muffins.
4. Multiply the remaining eggs by the price per egg to find the daily earnings.
</reasoning>
<answer>
54
</answer>

Extracted answer: 54
Gold answer:      18
Correct: False


In [13]:
# --- Full evaluation over all 16 test examples ---
correct_count = 0

for i, sample in enumerate(test_dataset):
    question    = get_question(sample)
    gold_answer = sample["answer"]
    model_output = generate_answer(question)
    pred_answer  = extract_answer(model_output)

    is_correct = normalize_answer(pred_answer) == normalize_answer(gold_answer)
    if is_correct:
        correct_count += 1

    status = "✓" if is_correct else "✗"
    print(f"[{status}] #{i+1:02d} | gold={gold_answer:>6}  pred={pred_answer:>6} | {question[:60]}...")

accuracy = correct_count / len(test_dataset) * 100
print("\n" + "=" * 60)
print(f"Accuracy: {accuracy:.1f}%  ({correct_count}/{len(test_dataset)} correct)")
print("Note: Only 30 GRPO steps — accuracy will improve with more training.")

[✗] #01 | gold=    18  pred=    38 | Janet’s ducks lay 16 eggs per day. She eats three for breakf...
[✓] #02 | gold=     3  pred=     3 | A robe takes 2 bolts of blue fiber and half that much white ...
[✗] #03 | gold= 70000  pred= 80000 | Josh decides to try flipping a house.  He buys a house for $...
[✓] #04 | gold=   540  pred=   540 | James decides to run 3 sprints 3 times a week.  He runs 60 m...
[✗] #05 | gold=    20  pred=    60 | Every day, Wendi feeds each of her chickens three cups of mi...
[✓] #06 | gold=    64  pred=    64 | Kylar went to the store to buy glasses for his new apartment...
[✓] #07 | gold=   260  pred=   260 | Toulouse has twice as many sheep as Charleston. Charleston h...
[✓] #08 | gold=   160  pred=   160 | Carla is downloading a 200 GB file. Normally she can downloa...
[✗] #09 | gold=    45  pred=   235 | John drives for 3 hours at a speed of 60 mph and then turns ...
[✗] #10 | gold=   460  pred=     3 | Eliza's rate per hour for the first 40 hours she works